# Análisis ConnectaTel



---
## Paso 1: Cargar y explorar


In [ ]:
# importar librerías
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# cargar archivos
plans = pd.read_csv('/datasets/plans.csv')
users = pd.read_csv('/datasets/users_latam.csv')
usage = pd.read_csv('/datasets/usage.csv')

In [ ]:
 # mostrar las primeras 5 filas de plans
plans.head(5)

In [ ]:
 # mostrar las primeras 5 filas de users
users.head(5)

In [ ]:
# mostrar las primeras 5 filas de usage
usage.head(5)

### 1.2 Exploración de la estructura de los datasets


In [ ]:
# revisar el número de filas y columnas de cada dataset
print("plans", plans.shape)
print("users", users.shape)
print("usage", usage.shape)

In [ ]:
# inspección de plans con .info()
plans.info()

In [ ]:
# inspección de users con .info()
users.info()

In [ ]:
# inspección de usage con .info()
usage.info()

---

## Paso 2: Identificación de problemas de calidad de datos

### 2.1 Revisión de valores nulos



In [ ]:
# cantidad de nulos para users
print(users.isna().sum())
print(users.isna().mean())

In [ ]:
# cantidad de nulos para usage
print(usage.isna().sum())
print(usage.isna().mean())

 ---

**Valores nulos**  
- ¿Qué columnas tienen valores faltantes y en qué proporción?
**Users:** Este dataset tiene dos columnas con valores faltantes: city (11.72%) y churn_date (88,35%).
**Usage:** Este data ser cuenta con 3 columnas con valores faltantes: date (0,12%), duration (55,19%) y lenght(44,74%)
- Indica qué harías: ¿imputar, eliminar, ignorar?
Para la primera tabla **user** se a ignoraran los nulos en **churn_date**, porque significa que los usuarios siguen con sus planes activos. Mientras que para **city** la decisión es imputar, ya que, es una de las variables categóricas centrales para poder segmentar los clientes y observar el comportamiento por lugar.
Para la segunda tabla, **date** los valores nulos se eliminarán, ya que solo son 0.12% lo cual es un número ínfimo de la información recolectada por la empresa. En cuento a **duration(55,19%)** y **lenght (44,74%)** se ignora y se investiga para observar si está alta cantidad de valores nulos esta relacionado a la variable 'type', pues quizás no existen datos porque la acción que se realizó no tiene relación con este tipo de variable, por ejemplo un 'text' no tiene 'duration', mientras que un 'call' no tiene un 'lenght' (esto se puede observar en usage.head(5)).

### 2.2 Detección de valores inválidos y sentinels



In [ ]:
# explorar columnas numéricas de users
columnas_numericas_users = ['user_id','age']
users[columnas_numericas_users].describe()

- La columna `user_id` muestra un comportamiento secuencial, lo cual es común en este tipo de datos que sirven de identificación.
- La columna `age` se observa un sentinels **-999**, lo que modifica por completo el análisis, es necesario reemplazarlo por 'pd.NA'.

In [ ]:
# explorar columnas numéricas de usage
columnas_numericas_usage=['id','user_id','duration','length']
usage[columnas_numericas_usage].describe()

- Las columnas `id` y `user_id` muestran un comportamiento esperado para ser llaves de identificación.
- Las columnas `duration` y `lenght` no muestran valores invalidos o sentinels negativos, como en el dataser anterior.

In [ ]:
# explorar columnas categóricas de users
columnas_user = ['city', 'plan']

for col in columnas_user:
    print(f"__Valores únicos para: {col}")
    print(users[col].value_counts(dropna=False))
    print()

- La columna `city` encontramos 469 datos catalogados como NaN, esto significa que son valores nulos, más 96 datos registrados bajo un ?. Ambas anomalías deben estandarizarse bajo la etiqueta de `unknown`
- La columna `plan` suma 4000 registros, que es la cantidad de datos que reune este dataset, no cuenta con errores de tipeo ni valores nulos.

In [ ]:

# explorar columna categórica de usage
print ("Valores único en 'type':")
print (usage['type'].unique())


- La columna `type` tiene las opciones 'call' y 'text'. No se encuentran datos mal escritos ni valores nulos.



---

**Valores inválidos o sentinels**  

**Users** `age` (Sentinel numérico) se encontro **-999** se recomienda reemplazar con la mediana del grupo, porque es una variable necesaria para la segmentación de clientes.
**Users** `city` (Sentinel categórico) se observaron dos columnas con valores invalidos, **Nan** y **?**, se recomienda crear una nueva columna con la etiqueta `Unknown`
En el dataset **Usage** no se encontraron valores inválidos ni sentinels.  

### 2.3 Revisión y estandarización de fechas



In [ ]:
# Convertir a fecha la columna `reg_date` de users
users['reg_date'] = pd.to_datetime(users['reg_date'],errors='coerce') # completa el código

In [ ]:
# Convertir a fecha la columna `date` de usage
usage['date'] = pd.to_datetime(usage['date'],errors='coerce') # completa el código

In [ ]:
# Revisar los años presentes en `reg_date` de users
print(users['reg_date'].dt.year.value_counts(dropna=False))


En `reg_date` se observa el año 2026, lo cual es imposible, ya que el dataset es hasta el 2024


In [ ]:
# Revisar los años presentes en `date` de usage
print(usage['date'].dt.year.value_counts(dropna=False))

En `date`, se observan 50 valores inválidos.

---
**Fechas fuera de rango**  

Para `reg_date` para los datos corruptos del 2026, los eliminaría.
Para `date` para los Nan, al orbservar que son 50 datos de 40.000 los eliminaría.
  



---

## Paso 3: Limpieza básica de datos

### 3.1 Corregir sentinels y fechas imposibles


In [ ]:
# Reemplazar -999 por la mediana de age
age_mediana = users['age'].median()
users['age'] = users['age'].replace(-999, age_mediana)

# Verificar cambios
users['age'].describe()

In [ ]:
# Reemplazar ? por NA en city
users['city'] = users['city'].replace('?', pd.NA)
# Verificar cambios
print("¿Sigue existiendo el '?' en city?:", "?" in users['city'].unique())

In [ ]:
# Marcar fechas futuras como NA para reg_date
users['reg_date'] = users['reg_date'].where(users['reg_date'].dt.year<=2024, pd.NaT)

# Verificar cambios
print("Distribución de años en reg_date tras la limpieza:")
print(users['reg_date'].dt.year.value_counts(dropna=False))


### 3.2 Corregir sentinels y fechas imposibles


In [ ]:
# Verificación MAR en usage (Missing At Random) para duration
usage.groupby('type')['duration'].apply(lambda x: x.isna().sum())

In [ ]:
# Verificación MAR en usage (Missing At Random) para length
usage.groupby('type')['length'].apply(lambda x: x.isna().sum())


En `duration` y `length` al agrupar por `type` se puede comprobar que los datos faltantes estan condicionados por el tipo de registro que se tiene, por lo que, son valores nulos por MAR, por lo que estos valores se mantienen, pues son reflejo del normal funcionamiento de la empresa.


## Paso 4: Summary statistics de uso por usuario


### 4.1 Agrupación por comportamiento de uso



In [ ]:

# Columnas auxiliares
usage["is_text"] = (usage["type"] == "text").astype(int) #conocer el total de mensajes
usage["is_call"] = (usage["type"] == "call").astype(int) #conocer el total de llamadas


# Agrupar información por usuario
usage_agg = usage.groupby('user_id').agg({
    "is_text": "sum",
    "is_call": "sum",
    "duration": "sum"
}).reset_index()

# observar resultado
usage_agg.head(3)


In [ ]:

# Renombrar columnas
usage_agg =usage_agg.rename(columns={
    "is_text" : "cant_mensajes",
    "is_call" : "cant_llamadas",
    "duration" : "cant_minutos_llamada"
})
# observar resultado
usage_agg.head(3)


In [ ]:

# Combinar la tabla agregada con el dataset de usuarios
user_profile = pd.merge(users, usage_agg, on="user_id", how="left")

# observar resultado
user_profile.head(5)


### 4.2 4.2 Resumen estadístico por usuario durante el 2024



In [ ]:
# Resumen estadístico de las columnas numéricas
columnas_num_perfil=['age','cant_mensajes','cant_llamadas','cant_minutos_llamada']
user_profile[columnas_num_perfil].describe()

In [ ]:
# Distribución porcentual del tipo de plan

user_profile['plan'].value_counts(normalize=True*100)


---

## 🧩Paso 5: Visualización de distribuciones (uso y clientes) y outliers


### 5.1 Visualización de Distribuciones


In [ ]:

# Histograma para visualizar la edad (age)
sns.histplot(data=user_profile,
             x='age',
             hue='plan',
             palette=["skyblue","green"],
             kde=True,
             multiple="stack"
)
plt.title("Distribución de edad de los usuarios según su Plan", fontsize=14,fontweight='bold')
plt.xlabel("Edad",fontsize=12)
plt.ylabel("Cantidad de usuarios", fontsize=12)
plt.show()


💡Insights:
- Distribución: se observa una distribución uniforme, es decir, no hay un segmento de edad que domine por sobre otras.
- Uso de plan: existe una mayor cantidad de usuarios en el plan basico.
- No existe una correlación clara entre edad y el tipo de plan que eligen los usuarios.

In [ ]:
# Histograma para visualizar la cant_mensajes
sns.histplot(
    data=user_profile,
    x="cant_mensajes",
    hue="plan",
    palette=["skyblue","green"],
    kde=True,
    multiple="stack"
)
plt.title("Distribución de Cantidad de Mensajes según su Plan", fontsize=14, fontweight='bold')
plt.xlabel("Cantidad de mensajes", fontsize=12)
plt.ylabel("Cantidad de usuarios", fontsize=12)
plt.show()


💡Insights:
- Distribución: Se observa una distribución normal (acampanada), la moda se observa en 5 mensajes, tanto para el plan básico, como para el premium
- los SMS no son tan utilizados por los clientes, ya que el número máximo de estos es de 17 (para un mes que tiene 30 días)

In [ ]:
# Histograma para visualizar la cant_llamadas
sns.histplot(
    data=user_profile,
    x="cant_llamadas",
    hue="plan",
    palette=["skyblue","green"],
    kde= True,
    multiple="stack"
)
plt.title("Distribución cantidad de llamadas según su plan", fontsize=14, fontweight='bold')
plt.xlabel("Cantidad de llamadas", fontsize=12)
plt.ylabel("Cantidad de usuarios", fontsize=12)
plt.show()


💡Insights:
- Distribución: se observa una distribución normal (acampanada) la mayor cantidad de usuarios realiza entre 2 y cuatro llamadas.
- Plan: El comportamiento es similar en ambos planes.

In [ ]:
# Histograma para visualizar la cant_minutos_llamada
sns.histplot(
    data=user_profile,
    x="cant_minutos_llamada",
    hue="plan",
    palette=["skyblue","green"],
    kde=True,
    multiple="stack"
)
plt.title("Distribución de minutos de llamada según su plan", fontsize=14, fontweight="bold")
plt.xlabel("Minutos de llamada", fontsize=12)
plt.ylabel("Cantidad de usuarios",fontsize=12)
plt.show()

💡Insights:
- Distribución: se puede observar claramente una distribución con sesgo a la derecha, lo que nos indica que existen outliers, que utilizan mayor cantidad de minutos al llamar.
- la mayor cantidad de usuarios utiliza entre 5 y 35 minutos de llamada.

### 5.2 Identificación de Outliers mediante Boxplots e IQR

In [ ]:

# Visualizando usando BoxPlot
columnas_numericas = ['age', 'cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']

for col in columnas_numericas:
    plt.figure(figsize=(6, 4))
    sns.boxplot(
           data=user_profile,
        x=col,
        y='plan',
        palette=["skyblue","green"]
    )
    plt.title(f'Boxplot:{col}', fontsize=12, fontweight='bold')
    plt.xlabel(col)
    plt.show()





💡Insights:
- Age: no presenta outliers
- cant_mensajes: presenta outliers a la derecha en ambos planes
- cant_llamadas: presenta outliers, en el plan básico presenta más outliers y más extremos, mientras que en el plan premium solo aparece un outlier.
- cant_minutos_llamada: como lo vimos en el histograma presenta outliers extremos en ambos planes y son a la derecha



In [ ]:
# Calcular límites con el método IQR
columnas_limites = ['cant_mensajes','cant_llamadas','cant_minutos_llamada']

#for para calcular IQR y límite superior

for col in columnas_limites:
    #cálculo Q1 y Q3
    Q1= user_profile[col].quantile(0.25)
    Q3= user_profile [col].quantile(0.75)

    #IQR
    iqr=Q3-Q1

    #Límite Superior
    limite_superior=Q3+1*iqr

    #Conteo cuántos outliers existen
    cantidad_outliers=(user_profile[col]>limite_superior).sum()

    #Impromir resultados
    print(f"Columna: {col}")
    print(f"-Q1: {Q1:.1f} | Q3: {Q3:.1f} | IQR: {iqr:.1f}")
    print(f"-Límite Superior de Outliers: {limite_superior:.2f}")
    print(f"-Cantidad de outliers detectados: {cantidad_outliers}")
    print(f"-Valor máximo registrado: {user_profile[col].max():.2f}")
    print("-" * 60)



In [ ]:

# Revisa los limites superiores y el max, para tomar la decisión de mantener los outliers o no
user_profile[columnas_limites].describe()




💡Insights:
- cant_mensajes: Mantener los outliers, porque el comportamiento es real, son 17 mensajes máximo en un mes, lo que no parece un comportamiento imposible.
- cant_llamadas: Mantener los outliers, porque al igual que el anterior son 15 llamadas máximo en el mes, lo cual no representa un comportamiento anómalo o imposible, sino que es como se comportan los usuarios de la empresa.
- cant_minutos_llamada: Mantener los outliers, se registra un máximo de 155.69 minutos por llamada, lo que aunque se aleja de la moda y la mediana, y modifica la media, es un comportamiento normal y que se debe tener en cuenta con esos usuarios, sobretodo para poder crear planes para ellos y poderlos segmentar debidamente.



---

## Paso 6: Segmentación de Clientes

### 6.1 Segmentación de Clientes Por Uso


In [ ]:
# Crear columna grupo_uso
condiciones_uso=[
    (user_profile["cant_llamadas"]<5) &(user_profile["cant_mensajes"]<5),

    (user_profile["cant_llamadas"]<10) & (user_profile["cant_mensajes"] <10)
]
opciones_uso=["Bajo uso","Uso medio"]
user_profile['grupo_uso']=np.select(condiciones_uso, opciones_uso, default="Alto uso")


In [ ]:

# verificar cambios
user_profile.head()


### 6.2 Segmentación de Clientes Por Edad



In [ ]:
# Crear columna grupo_edad
condiciones_edad=[
    (user_profile["age"]<30),
    (user_profile["age"]<60)
]
opciones_edad=['Joven','Adulto']

user_profile["grupo_edad"]=np.select(condiciones_edad, opciones_edad, default='Adulto Mayor')

In [ ]:
# verificar cambios
user_profile.head()

### 6.3 Visualización de la Segmentación de Clientes


In [ ]:
# Visualización de los segmentos por uso
sns.countplot(
    data=user_profile,
    x='grupo_uso',
    order=['Bajo uso','Uso medio','Alto uso'],
    palette=['royalblue','green','skyblue']
)
plt.title("Distribución de usuarios según su grupo de uso", fontsize=14, fontweight= "bold")
plt.xlabel("Grupo de uso",fontsize=12)
plt.ylabel("Cantidad de Usuarios", fontsize=12)
plt.show()

In [ ]:
# Visualización de los segmentos por edad
sns.countplot(
    data=user_profile,
    x="grupo_edad",
    order=['Joven','Adulto','Adulto Mayor'],
    palette=['royalblue','green','skyblue']
)
plt.title("Distribución de usuarios según su edad", fontsize=14,fontweight="bold")
plt.xlabel("Segmentación por edad")
plt.ylabel("Cantidad de usuarios")
plt.show()



---
## Paso 7: Insight Ejecutivo para Stakeholders

- ¿Qué problemas tenían originalmemte los datos?¿Qué porcentaje, o cantidad de filas, de esa columna representaban?
El primer problema que se encontró en los datos fue el valor -999 en la columna de edad, lo cual fue reemplazado con la mediana de la columna. Ademas se encontró un año imposible porque los datos eran hasta el 2024 y aparecía el 2026, lo cual se capó. Se tuvo que hacer una investigación exhaustiva de la tabla usage, pues existía gran cantidad de valores inválidos en 'duration' y 'length', se llego a la conclusión de que no se trataba de errores de medición, sino de datos ausentes al azar (MAR), se trataba de un "error" que reflejaba el normal funcionamiento de la empresa

- ¿Qué segmentos de clientes identificaste y cómo se comportan según su edad y nivel de uso?

-->Segmentación por edad:
  - Jóvenes (<30): son el grupo más pequeño (750 usuarios aprox.) tienen hábitos de consumo balanceados y se distribuyen equitativamente en el uso de planes.
  - Adultos (30-59): son el grupo con mayor numero de usuarios (2020 usuarios aprox), es el segmento más activo en todos los servicios (mensajeria, llamada)
  - Adultos Mayores (>=60): Son el segundo grupo de usuarios (1230 usuarios) mantienen un patron de uso constante y al igual que los dos grupos anteriores existe una distribución uniforme en el uso de los planes.

-->Segmentación por nivel de uso:
    -Bajo uso: es un grupo de 780 personas que tienen consumos mínimos de llamadas y mensaje (<5)
    -Uso Medio: Este grupo es el que más usuarios abarca, la mayoria tienen un uso moderado de los servicios de la empresa.
    -Alto Uso: Estos son "súper usuarios" generan traficos intensos en la red de llamadas sobretodo, son aproximadamente 270 usuarios.

- ¿Qué segmentos parecen más valiosos para ConnectaTel y por qué?

- El segmento "adulto mayor": son el segundo grupo de segmentación etaria y tienen un uso uniforme que los demás grupos, pero es normalmente un nicho fiel y con bajo riesgo de abandono

- El segmento de "alto uso": es un grupo pequeño, pero se pueden generar planes para ellos, ya que generan mayor tráfico y rentabilidad.
- ¿Qué patrones de uso extremo (outliers) encontraste y qué implican para el negocio?
    -Mensajes: 103 casos por encima del límite de 10 mensajes, con un máximo de 17.
    -Llamadas: 70 casos sobre el límite de 9 llamadas, llegando hasta 15
    -Minutos de llamada: El hallazgo más notables, 207 outliers superando los 51.71, con un máximo de 155.69 minutos de llamada.

Estos datos al no ser errores del sistema, son usuarios de alto consumo. Su existencia confirma que hay una demanda del servicio de llamada. Algo que la empresa debe tener en cuenta al dimensionar la gestión y logística de su servicio. Además es relevante hablar sobre si realmente sigue siendo un plus hacer gastos publicitarios sobre los mensajes ilimitados, ya que los usuarios no los usan.

- ¿Qué recomendaciones harías para mejorar la oferta actual de planes o crear nuevos planes basados en los segmentos y patrones detectados?
- Hacer un plan senior, tarifas de llamadas y conectividad diferenciada, con ayuda tecnológica y atención al cliente prioritaria. Para poder generar lealtad con este segmento que además de ser amplio, es estable.


### Análisis ejecutivo

⚠️ **Problemas detectados en los datos**
- `'Age'`: se detectó un 2% (80 filas) con valor -999. Se corrigió mediante el reemplazo con la mediana de la columna.
- `'Valores Nulos'`: se identificaron 22076 nulos en `'duration'` en `'mensajes'` y 17896 nulos en `'lenght'` en la columna `'llamada'`. Son nulos tipo MAR y refleja la operación normal del servicio entregado. Se mantuvieron.

🔍 **Segmentos por Edad**
- `'Demografía del servicio'`: la base se compone por `'Adultos'` (50.5%), `'Adultos Mayores'` (30,7%) y `'Jóvenes'` (18,7%.)
- La tasa de contratación del plan premium se distribuye de forma equitativa en todas las edades. Lo que quiere decir que la edad no es una variable causal para la elección del plan.


📊 **Segmentos por Nivel de Uso**
- La mayor parte de los usuarios, tiene un `'Uso medio'` (73,7%), seguida por `'Bajo uso'` (19.5%) y `'Alto uso'` (6,8%)

📊**Outliers**
- Se detectaron outiliers de consumo únicamente a la derecha (límite superior): 103 en mensajes, 70 en llamadas y 207 minutos. Esto representa los Heavy Users del servicio y ayudan a observar la capacidad real de la red.
  
➡️ Esto sugiere que:
- La edad no es una barrera digital, los `'Adultos Mayores'` no estan excluidos del mundo digital, son mas activos que los jóvenes, ya que demandan conectividad, por lo que, se pueden hacer campañas específicas para fidelizarlos con la marca.
- Los mensajes de texto no generan un valor por sí mismo, van en retirada, lo que empuja a los usuarios a ser premium no va por "SMS ilimitados".

💡 **Recomendaciones**
- Rediseño "plan premium": eliminar de la campaña los "SMS ilimitados (la moda es de 5 mensajes) y reenfocarlo en datos moviles y bolsas de minutos de larga duración.
- Crear un plan "Senior": Lanzar productos focalizados a los adultos mayores, ofreciendo tarifas competitivas de voz y atención al cliente prioritario, además de posible alfabetización tecnológica o apoyo tecnológico.